# Project 30 — LongevityAI: Biological Age & Healthy-Aging Risk Intelligence

## Human biomarker analytics for geroscience, longevity research and healthy-aging decision support

This project converts **real NHANES III human biomarker data + linked mortality outcomes** into an end-to-end longevity analytics system.

### Core question
Can a biomarker-derived **Phenotypic Age** measure identify physiological aging differences beyond chronological age, and can we turn those differences into interpretable population-level research signals?

### What this notebook demonstrates
- real human clinical biomarker data
- biological / phenotypic age calculation from published coefficients
- age-acceleration analysis
- aging-related mortality discrimination
- machine-learning risk modelling
- calibration, ROC/PR analysis and risk deciles
- biomarker explainability and sensitivity analysis
- healthy-aging phenotype segmentation
- subgroup robustness checks
- a reusable participant scoring function

> **Important:** this is a research/portfolio project, not a medical diagnosis, treatment recommendation, or claim that aging can be reversed.


## Scientific basis and data provenance

**Phenotypic Age (PhenoAge)** was developed to capture mortality-related physiological dysregulation using chronological age plus nine routine clinical biomarkers.

Published work: Liu Z, Kuo P-L, Horvath S, Crimmins E, Ferrucci L, Levine M. *A new aging measure captures morbidity and mortality risk across diverse subpopulations from NHANES IV.* PLOS Medicine (2018).

Data:
- NHANES III cohort biomarker extract: public human health survey data.
- Public-use linked mortality outcomes derived from NCHS / National Death Index linkage.
- Reproducible cohort mirror used here: https://github.com/nopara73/PhenoAge2/tree/master/nhanes3-phenoage

The public-use mortality files may contain perturbed follow-up time / cause-of-death information for disclosure protection, so this notebook treats survival-related results as **population research signals**, not individual clinical predictions.


In [ ]:
import warnings, math, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve,
    precision_recall_curve, classification_report,
    ConfusionMatrixDisplay, brier_score_loss
)
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
print("LongevityAI environment ready ✅")


## 1. Load the real NHANES III biomarker and mortality data

In [ ]:
COHORT_URL = "https://raw.githubusercontent.com/nopara73/PhenoAge2/master/nhanes3-phenoage/cohort.csv"
OUTCOMES_URL = "https://raw.githubusercontent.com/nopara73/PhenoAge2/master/nhanes3-phenoage/outcomes.csv"

cohort = pd.read_csv(COHORT_URL)
outcomes = pd.read_csv(OUTCOMES_URL)

print("Biomarker rows:", f"{len(cohort):,}")
print("Outcome rows:", f"{len(outcomes):,}")
display(cohort.head())
display(outcomes.head())


## 2. Audit identifiers, missingness and outcome prevalence

In [ ]:
assert cohort["SEQN"].is_unique
assert outcomes["SEQN"].is_unique

audit = pd.DataFrame({
    "dtype": cohort.dtypes.astype(str),
    "missing_n": cohort.isna().sum(),
    "missing_pct": cohort.isna().mean().mul(100)
}).sort_values("missing_pct", ascending=False)

display(audit)
print("Mortality prevalence:", outcomes["mortstat"].mean())
print("Aging-related event prevalence:", outcomes["aging_related_event"].mean())


## 3. Join baseline biomarkers to linked outcomes

In [ ]:
df = cohort.merge(outcomes, on="SEQN", how="inner", validate="one_to_one")
print("Joined participants:", f"{len(df):,}")
print("All-cause deaths:", int(df["mortstat"].sum()))
print("Aging-related events:", int(df["aging_related_event"].sum()))
display(df.head())


## 4. Map raw NHANES fields to clinically readable features

The PhenoAge implementation expects:
- albumin: g/L
- creatinine: µmol/L
- glucose: mmol/L
- CRP: mg/L before natural log
- lymphocytes: %
- MCV: fL
- RDW: %
- alkaline phosphatase: U/L
- WBC: 10³ cells/µL
- chronological age: years


In [ ]:
work = df.copy()

work["chronological_age"] = work["HSAGEIR"].astype(float)
work["albumin_gL"] = work["AMP"].astype(float) * 10.0
work["creatinine_umolL"] = work["CEP"].astype(float) * 88.4
work["glucose_mmolL"] = work["SGP"].astype(float) / 18.0182
work["crp_mgL"] = work["CRP"].astype(float) * 10.0
work["ln_crp"] = np.log(work["crp_mgL"].clip(lower=1e-8))
work["lymphocyte_pct"] = work["LMPPCNT"].astype(float)
work["mcv_fL"] = work["MVPSI"].astype(float)
work["rdw_pct"] = work["RWP"].astype(float)
work["alp_UL"] = work["APPSI"].astype(float)
work["wbc_10e3_uL"] = work["WCP"].astype(float)

biomarkers = [
    "albumin_gL", "creatinine_umolL", "glucose_mmolL", "ln_crp",
    "lymphocyte_pct", "mcv_fL", "rdw_pct", "alp_UL", "wbc_10e3_uL"
]

display(work[["chronological_age"] + biomarkers].describe().T)


## 5. Implement the published Phenotypic Age equation

In [ ]:
def calculate_phenoage(frame: pd.DataFrame) -> pd.Series:
    xb = (
        -19.90667
        - 0.03359355 * frame["albumin_gL"]
        + 0.009506491 * frame["creatinine_umolL"]
        + 0.1953192 * frame["glucose_mmolL"]
        + 0.09536762 * frame["ln_crp"]
        - 0.01199984 * frame["lymphocyte_pct"]
        + 0.02676401 * frame["mcv_fL"]
        + 0.3306156 * frame["rdw_pct"]
        + 0.001868778 * frame["alp_UL"]
        + 0.05542406 * frame["wbc_10e3_uL"]
        + 0.08035356 * frame["chronological_age"]
    )

    mortality_component = 1.51714 * np.exp(xb) / 0.007692696
    phenoage = 141.50225 + np.log(0.0055305 * mortality_component) / 0.090165
    return phenoage

work["phenoage"] = calculate_phenoage(work)
work["phenoage_advance"] = work["phenoage"] - work["chronological_age"]

assert np.isfinite(work["phenoage"]).all()
print(work[["chronological_age", "phenoage", "phenoage_advance"]].describe())


## 6. Calculate age-adjusted Phenotypic Age Acceleration

In [ ]:
age_model = LinearRegression()
age_model.fit(work[["chronological_age"]], work["phenoage"])
expected_phenoage = age_model.predict(work[["chronological_age"]])

work["phenoage_accel"] = work["phenoage"] - expected_phenoage

print("Mean residual acceleration:", work["phenoage_accel"].mean())
print("Correlation with chronological age:",
      np.corrcoef(work["chronological_age"], work["phenoage_accel"])[0, 1])


## 7. Chronological age vs Phenotypic Age

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(work["chronological_age"], work["phenoage"], s=10, alpha=0.25)
lo = min(work["chronological_age"].min(), work["phenoage"].min())
hi = max(work["chronological_age"].max(), work["phenoage"].max())
ax.plot([lo, hi], [lo, hi], linestyle="--", label="Phenotypic age = chronological age")
ax.set(title="Chronological Age vs Phenotypic Age",
       xlabel="Chronological age (years)", ylabel="Phenotypic age (years)")
ax.legend()
plt.show()

print("Correlation:", np.corrcoef(work["chronological_age"], work["phenoage"])[0, 1])


## 8. Distribution of accelerated vs decelerated aging profiles

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(work["phenoage_accel"], bins=50, alpha=0.8)
ax.axvline(0, linestyle="--")
ax.set(title="Phenotypic Age Acceleration",
       xlabel="Residual PhenoAge acceleration (years)", ylabel="Participants")
plt.show()

work["aging_profile"] = pd.cut(
    work["phenoage_accel"],
    bins=[-np.inf, -5, -1, 1, 5, np.inf],
    labels=["Much slower", "Slower", "Age-aligned", "Faster", "Much faster"]
)

display(work["aging_profile"].value_counts(sort=False).to_frame("participants"))


## 9. Compare aging-related event rates across aging profiles

In [ ]:
profile_summary = (
    work.groupby("aging_profile", observed=True)
        .agg(
            participants=("SEQN", "size"),
            mean_age=("chronological_age", "mean"),
            mean_phenoage=("phenoage", "mean"),
            mean_acceleration=("phenoage_accel", "mean"),
            aging_event_rate=("aging_related_event", "mean"),
            all_cause_mortality_rate=("mortstat", "mean")
        )
)

display(profile_summary)

fig, ax = plt.subplots(figsize=(9, 5))
profile_summary["aging_event_rate"].plot(kind="bar", ax=ax)
ax.set(title="Aging-Related Event Rate by Phenotypic Aging Profile",
       xlabel="Phenotypic aging profile", ylabel="Observed event rate")
plt.xticks(rotation=25, ha="right")
plt.show()


## 10. Biomarker correlation map

In [ ]:
corr_cols = ["chronological_age", "phenoage", "phenoage_accel"] + biomarkers
corr = work[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
im = ax.imshow(corr.values, aspect="auto")
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=90)
ax.set_yticks(range(len(corr.index)), corr.index)
ax.set_title("Longevity Biomarker Correlation Matrix")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


## 11. Benchmark: chronological age vs Phenotypic Age

In [ ]:
y = work["aging_related_event"].astype(int)

score_table = pd.DataFrame([
    {
        "score": "Chronological age",
        "ROC_AUC": roc_auc_score(y, work["chronological_age"]),
        "PR_AUC": average_precision_score(y, work["chronological_age"])
    },
    {
        "score": "Phenotypic age",
        "ROC_AUC": roc_auc_score(y, work["phenoage"]),
        "PR_AUC": average_precision_score(y, work["phenoage"])
    },
    {
        "score": "PhenoAge acceleration",
        "ROC_AUC": roc_auc_score(y, work["phenoage_accel"]),
        "PR_AUC": average_precision_score(y, work["phenoage_accel"])
    }
]).sort_values("ROC_AUC", ascending=False)

display(score_table)


## 12. ROC curves for age-based longevity signals

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for label, score in {
    "Chronological age": work["chronological_age"],
    "Phenotypic age": work["phenoage"],
    "PhenoAge acceleration": work["phenoage_accel"],
}.items():
    fpr, tpr, _ = roc_curve(y, score)
    auc = roc_auc_score(y, score)
    ax.plot(fpr, tpr, label=f"{label} — AUC {auc:.3f}")

ax.plot([0, 1], [0, 1], linestyle="--")
ax.set(title="Aging-Related Mortality Discrimination",
       xlabel="False-positive rate", ylabel="True-positive rate")
ax.legend()
plt.show()


## 13. Risk-decile gradient

In [ ]:
work["phenoage_decile"] = pd.qcut(work["phenoage"], 10, labels=False, duplicates="drop") + 1

deciles = (
    work.groupby("phenoage_decile")
        .agg(
            participants=("SEQN", "size"),
            mean_phenoage=("phenoage", "mean"),
            mean_age=("chronological_age", "mean"),
            event_rate=("aging_related_event", "mean")
        )
)

display(deciles)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(deciles.index, deciles["event_rate"], marker="o")
ax.set(title="Observed Aging-Related Event Rate by PhenoAge Decile",
       xlabel="Phenotypic Age decile (1 = lowest)", ylabel="Observed event rate")
ax.grid(alpha=0.25)
plt.show()

risk_ratio = deciles.iloc[-1]["event_rate"] / deciles.iloc[0]["event_rate"]
print(f"Top-vs-bottom decile event-rate ratio: {risk_ratio:.2f}x")


## 14. Build leakage-safe train/test data

In [ ]:
model_features = ["chronological_age"] + biomarkers
X = work[model_features].copy()
y = work["aging_related_event"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Test event prevalence:", y_test.mean())


## 15. Logistic-regression biomarker model

In [ ]:
logit = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=4000, class_weight="balanced", random_state=SEED))
])

logit.fit(X_train, y_train)
p_logit = logit.predict_proba(X_test)[:, 1]

print("Logistic ROC-AUC:", roc_auc_score(y_test, p_logit))
print("Logistic PR-AUC:", average_precision_score(y_test, p_logit))
print("Logistic Brier:", brier_score_loss(y_test, p_logit))


## 16. Non-linear Random Forest benchmark

In [ ]:
forest = RandomForestClassifier(
    n_estimators=500,
    min_samples_leaf=10,
    class_weight="balanced_subsample",
    random_state=SEED,
    n_jobs=-1
)

forest.fit(X_train, y_train)
p_forest = forest.predict_proba(X_test)[:, 1]

model_results = pd.DataFrame([
    {
        "model": "Chronological age only",
        "ROC_AUC": roc_auc_score(y_test, X_test["chronological_age"]),
        "PR_AUC": average_precision_score(y_test, X_test["chronological_age"])
    },
    {
        "model": "Logistic biomarkers",
        "ROC_AUC": roc_auc_score(y_test, p_logit),
        "PR_AUC": average_precision_score(y_test, p_logit)
    },
    {
        "model": "Random Forest biomarkers",
        "ROC_AUC": roc_auc_score(y_test, p_forest),
        "PR_AUC": average_precision_score(y_test, p_forest)
    },
]).sort_values("ROC_AUC", ascending=False)

display(model_results)


## 17. Compare ROC and precision-recall curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for label, score in {
    "Age only": X_test["chronological_age"].values,
    "Logistic": p_logit,
    "Random Forest": p_forest
}.items():
    fpr, tpr, _ = roc_curve(y_test, score)
    ax.plot(fpr, tpr, label=f"{label} AUC={roc_auc_score(y_test, score):.3f}")
ax.plot([0, 1], [0, 1], linestyle="--")
ax.set(title="Model ROC Comparison", xlabel="False-positive rate", ylabel="True-positive rate")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
for label, score in {
    "Age only": X_test["chronological_age"].values,
    "Logistic": p_logit,
    "Random Forest": p_forest
}.items():
    precision, recall, _ = precision_recall_curve(y_test, score)
    ax.plot(recall, precision, label=f"{label} AP={average_precision_score(y_test, score):.3f}")
ax.set(title="Precision–Recall Comparison", xlabel="Recall", ylabel="Precision")
ax.legend()
plt.show()


## 18. Calibration audit

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for label, probs in {"Logistic": p_logit, "Random Forest": p_forest}.items():
    frac_pos, mean_pred = calibration_curve(y_test, probs, n_bins=10, strategy="quantile")
    ax.plot(mean_pred, frac_pos, marker="o", label=label)

ax.plot([0, 1], [0, 1], linestyle="--")
ax.set(title="Calibration — Predicted vs Observed Aging-Related Event Risk",
       xlabel="Mean predicted probability", ylabel="Observed event rate")
ax.legend()
plt.show()


## 19. Explainability with permutation importance

In [ ]:
perm = permutation_importance(
    forest, X_test, y_test,
    scoring="roc_auc", n_repeats=10,
    random_state=SEED, n_jobs=-1
)

importance = pd.DataFrame({
    "feature": model_features,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

display(importance)

fig, ax = plt.subplots(figsize=(9, 6))
plot_imp = importance.sort_values("importance_mean")
ax.barh(plot_imp["feature"], plot_imp["importance_mean"])
ax.set(title="Held-Out Permutation Importance", xlabel="Mean decrease in ROC-AUC")
plt.show()


## 20. Linear-model directionality

In [ ]:
coef = logit.named_steps["model"].coef_[0]

coef_table = pd.DataFrame({
    "feature": model_features,
    "standardised_coefficient": coef
}).sort_values("standardised_coefficient", key=np.abs, ascending=False)

display(coef_table)

fig, ax = plt.subplots(figsize=(9, 6))
plot_coef = coef_table.sort_values("standardised_coefficient")
ax.barh(plot_coef["feature"], plot_coef["standardised_coefficient"])
ax.axvline(0)
ax.set(title="Logistic Model — Standardised Feature Direction", xlabel="Coefficient")
plt.show()


## 21. Healthy-aging phenotype segmentation

In [ ]:
cluster_features = biomarkers.copy()
Z = StandardScaler().fit_transform(work[cluster_features])

silhouette_rows = []
for k in range(2, 7):
    labels = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit_predict(Z)
    silhouette_rows.append({"k": k, "silhouette": silhouette_score(Z, labels)})

silhouette_df = pd.DataFrame(silhouette_rows)
display(silhouette_df)

best_k = int(silhouette_df.loc[silhouette_df["silhouette"].idxmax(), "k"])
print("Selected K:", best_k)


## 22. Fit and visualise the aging phenotypes

In [ ]:
kmeans = KMeans(n_clusters=best_k, n_init=30, random_state=SEED)
work["aging_cluster"] = kmeans.fit_predict(Z)

pca = PCA(n_components=2, random_state=SEED)
xy = pca.fit_transform(Z)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(xy[:, 0], xy[:, 1], c=work["aging_cluster"], s=10, alpha=0.4)
ax.set(title=f"Unsupervised Healthy-Aging Phenotypes (K={best_k})",
       xlabel="PCA component 1", ylabel="PCA component 2")
plt.show()

print("PCA variance explained:", pca.explained_variance_ratio_.sum())


## 23. Profile the discovered phenotypes

In [ ]:
cluster_profile = (
    work.groupby("aging_cluster")
        .agg(
            participants=("SEQN", "size"),
            mean_age=("chronological_age", "mean"),
            mean_phenoage=("phenoage", "mean"),
            mean_accel=("phenoage_accel", "mean"),
            aging_event_rate=("aging_related_event", "mean"),
            all_cause_mortality=("mortstat", "mean")
        )
)

display(cluster_profile)

biomarker_profile = work.groupby("aging_cluster")[biomarkers].median().T
display(biomarker_profile)


## 24. Age-band robustness analysis

In [ ]:
work["age_band"] = pd.cut(
    work["chronological_age"],
    bins=[19, 39, 49, 59, 69, 79, np.inf],
    labels=["20–39", "40–49", "50–59", "60–69", "70–79", "80+"]
)

rows = []
for band, g in work.groupby("age_band", observed=True):
    if g["aging_related_event"].nunique() < 2:
        continue
    rows.append({
        "age_band": str(band),
        "n": len(g),
        "events": int(g["aging_related_event"].sum()),
        "PhenoAge_AUC": roc_auc_score(g["aging_related_event"], g["phenoage"]),
        "Accel_AUC": roc_auc_score(g["aging_related_event"], g["phenoage_accel"])
    })

subgroup_auc = pd.DataFrame(rows)
display(subgroup_auc)


## 25. One-at-a-time biomarker sensitivity

This is **not an intervention-effect estimate**. It asks only how sensitive the published equation is to small input changes around a reference profile.


In [ ]:
reference = work[["chronological_age"] + biomarkers].median().to_frame().T
base = float(calculate_phenoage(reference).iloc[0])

sensitivity_rows = []
for col in biomarkers:
    perturbed = reference.copy()
    delta = 0.10 if col == "ln_crp" else 0.10 * work[col].std()
    perturbed[col] = perturbed[col] + delta
    changed = float(calculate_phenoage(perturbed).iloc[0])
    sensitivity_rows.append({
        "biomarker": col,
        "perturbation": delta,
        "PhenoAge_change_years": changed - base
    })

sensitivity = pd.DataFrame(sensitivity_rows).sort_values(
    "PhenoAge_change_years", key=np.abs, ascending=False
)

display(sensitivity)

fig, ax = plt.subplots(figsize=(9, 6))
p = sensitivity.sort_values("PhenoAge_change_years")
ax.barh(p["biomarker"], p["PhenoAge_change_years"])
ax.axvline(0)
ax.set(title="Equation Sensitivity Around a Median Reference Profile",
       xlabel="Change in calculated PhenoAge (years)")
plt.show()


## 26. Reusable individual scoring function

In [ ]:
def longevity_score(
    chronological_age,
    albumin_gL,
    creatinine_umolL,
    glucose_mmolL,
    crp_mgL,
    lymphocyte_pct,
    mcv_fL,
    rdw_pct,
    alp_UL,
    wbc_10e3_uL
):
    row = pd.DataFrame([{
        "chronological_age": chronological_age,
        "albumin_gL": albumin_gL,
        "creatinine_umolL": creatinine_umolL,
        "glucose_mmolL": glucose_mmolL,
        "ln_crp": np.log(max(crp_mgL, 1e-8)),
        "lymphocyte_pct": lymphocyte_pct,
        "mcv_fL": mcv_fL,
        "rdw_pct": rdw_pct,
        "alp_UL": alp_UL,
        "wbc_10e3_uL": wbc_10e3_uL
    }])

    pa = float(calculate_phenoage(row).iloc[0])
    gap = pa - chronological_age

    return {
        "chronological_age": round(float(chronological_age), 1),
        "phenotypic_age": round(pa, 1),
        "raw_age_gap": round(gap, 1),
        "interpretation": "population-level research score only; not a diagnosis or treatment recommendation"
    }

example = longevity_score(
    chronological_age=50,
    albumin_gL=44,
    creatinine_umolL=80,
    glucose_mmolL=5.2,
    crp_mgL=1.0,
    lymphocyte_pct=30,
    mcv_fL=90,
    rdw_pct=12.5,
    alp_UL=70,
    wbc_10e3_uL=6.0
)

example


## 27. Example cohort records: actual vs phenotypic age

In [ ]:
example_people = (
    work[[
        "SEQN", "chronological_age", "phenoage",
        "phenoage_advance", "phenoage_accel",
        "aging_related_event"
    ]]
    .sample(12, random_state=SEED)
    .sort_values("chronological_age")
)

display(example_people)


## 28. Data-quality and reproducibility checks

In [ ]:
assert work["SEQN"].is_unique
assert work["chronological_age"].between(20, 120).all()
assert np.isfinite(work[biomarkers + ["phenoage", "phenoage_accel"]].to_numpy()).all()
assert set(np.unique(work["aging_related_event"])).issubset({0, 1})
assert X_train.index.intersection(X_test.index).empty
assert len(model_results) == 3
assert work["aging_cluster"].nunique() == best_k

print("All LongevityAI integrity checks passed ✅")


## 29. External reproducibility benchmark

The source cohort's validation report records:
- 9,358 participants
- 3,797 all-cause deaths
- 2,710 aging-related deaths
- PhenoAge ROC-AUC: 0.8534 for the source binary aging-related mortality benchmark

This notebook independently recalculates the score and adds train/test ML benchmarking, calibration, phenotype segmentation, explainability, subgroup analysis and sensitivity analysis.


## 30. Longevity / anti-aging industry interpretation

### Potential research uses
1. Biomarker intelligence from routine laboratory measurements.
2. Cohort stratification for healthy-aging and geroscience studies.
3. Exploratory trial endpoints alongside validated clinical outcomes.
4. Longitudinal monitoring if repeated measurements are available.
5. Benchmarking whether new omics, imaging or wearable features add value.

### What this project does not prove
- that a supplement, drug or lifestyle intervention reverses aging;
- that PhenoAge is a diagnosis;
- that an individual will live to a particular age;
- that observational feature importance is causal.


## 31. Final executive summary

In [ ]:
best = model_results.iloc[0]
top_feature = importance.iloc[0]["feature"]
median_gap = work["phenoage_advance"].median()
fast_pct = (work["phenoage_accel"] > 0).mean() * 100

print("PROJECT 30 — LONGEVITYAI: BIOLOGICAL AGE & HEALTHY-AGING INTELLIGENCE")
print("=" * 78)
print(f"Human participants analysed: {len(work):,}")
print(f"Aging-related outcome events: {int(work['aging_related_event'].sum()):,}")
print(f"Median raw PhenoAge gap: {median_gap:+.2f} years")
print(f"Participants above age-adjusted expected PhenoAge: {fast_pct:.1f}%")
print()
print("MODEL BENCHMARK")
print("-" * 78)
print(f"Best held-out model: {best['model']}")
print(f"ROC-AUC: {best['ROC_AUC']:.4f} | PR-AUC: {best['PR_AUC']:.4f}")
print(f"Most influential held-out biomarker feature: {top_feature}")
print()
print("SOLUTION")
print("-" * 78)
print("Use biological-age analytics as a geroscience research and cohort-stratification")
print("layer, benchmark it against chronological age, and validate any intervention")
print("prospectively before making health or anti-aging claims.")
print()
print("PROJECT 30 COMPLETE ✅")
